In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/kidney_disease.csv')

In [2]:
#clean column names
df.columns = df.columns.str.strip().str.lower()

In [3]:
#fix dirty values
df.replace(['\t?', '?'], np.nan, inplace=True)

,id,age,blood pressure,specific gravity,albumin,sugar,red blood cells,pus cell,pus cell clumps,bacteria,...,packed cell volume,white blood cell count,red blood cell count,hypertension,diabetes mellitus,coronary artery disease,appetite,pedal edema,anemia,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,395,55.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,...,47.0,6700.0,4.9,no,no,no,good,no,no,notckd
396,396,42.0,70.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,...,54.0,7800.0,6.2,no,no,no,good,no,no,notckd
397,397,12.0,80.0,1.020,0.0,0.0,normal,normal,notpresent,notpresent,...,49.0,6600.0,5.4,no,no,no,good,no,no,notckd
398,398,17.0,60.0,1.025,0.0,0.0,normal,normal,notpresent,notpresent,...,51.0,7200.0,5.9,no,no,no,good,no,no,notckd


In [4]:
cols_to_convert = ['packed cell volume', 'white blood cell count', 'red blood cell count']

for col in cols_to_convert:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 26 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       400 non-null    str    
 1   age                      376 non-null    float64
 2   blood pressure           376 non-null    float64
 3   specific gravity         341 non-null    float64
 4   albumin                  342 non-null    float64
 5   sugar                    340 non-null    float64
 6   red blood cells          243 non-null    str    
 7   pus cell                 324 non-null    str    
 8   pus cell clumps          381 non-null    str    
 9   bacteria                 381 non-null    str    
 10  blood glucose random     343 non-null    float64
 11  blood urea               367 non-null    float64
 12  serum creatinine         369 non-null    float64
 13  sodium                   304 non-null    float64
 14  potassium                303 non-null

In [6]:
if 'id' in df.columns:
    df = df.drop('id', axis=1)

In [7]:
#separate features and target
X = df.drop('classification', axis=1)
y = df['classification']

In [8]:
#encode target
y = y.map({'ckd': 1, 'notckd': 0})

In [9]:
num_cols = X.select_dtypes(include=['float64', 'int64']).columns
cat_cols = X.select_dtypes(include=['object','str']).columns

In [10]:
#handling missing values
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
X[cat_cols] = X[cat_cols].fillna(X[cat_cols].mode().iloc[0])

In [11]:
#encode categorial features
from sklearn.preprocessing import LabelEncoder

le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le

In [12]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   age                      400 non-null    float64
 1   blood pressure           400 non-null    float64
 2   specific gravity         400 non-null    float64
 3   albumin                  400 non-null    float64
 4   sugar                    400 non-null    float64
 5   red blood cells          400 non-null    int64  
 6   pus cell                 400 non-null    int64  
 7   pus cell clumps          400 non-null    int64  
 8   bacteria                 400 non-null    int64  
 9   blood glucose random     400 non-null    float64
 10  blood urea               400 non-null    float64
 11  serum creatinine         400 non-null    float64
 12  sodium                   400 non-null    float64
 13  potassium                400 non-null    float64
 14  hemoglobin               400 non-null

In [13]:
#feature scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [14]:
#save preprocessing models for Django use
import pickle

pickle.dump(scaler, open('../models/scaler.pkl', 'wb'))
pickle.dump(le_dict, open('../models/encoder.pkl', 'wb'))

In [15]:
#save feature order
feature_columns = X.columns.tolist()

pickle.dump(feature_columns, open('../models/features.pkl', 'wb'))